# 피처 그룹 Ablation 실험 (TimeSeriesSplit 버전)
- **고정**: ID (sector + ticker OHE 125개)
- **조합**: FIN / MKT / EMO / ECON → 15가지 조합
- **FIN 버전**: vol3d / vol5d
- **라벨 임계값**: ±0.5% / ±0.7% / ±1.0%
- **모델**: HistGradientBoostingClassifier
- **평가**: TimeSeriesSplit CV (날짜 기준 **3split**, 항상 과거→미래) + Holdout 마지막 8거래일
- **3-fold 채택 이유**: 약 36거래일의 소규모 데이터에서 5-fold는 초기 split의 학습 구간이 6일에 불과해 평가가 불안정. 3-fold는 fold당 학습·검증량(9일+)을 확보해 더 안정적.
- **핵심**: 일반 GroupKFold와 달리 학습 구간이 항상 검증 구간보다 과거임을 보장 (시계열 누수 차단)

In [ ]:
import pandas as pd
import numpy as np
import itertools
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score

In [ ]:
# ── 설정 ──────────────────────────────────────────────
HOLDOUT_DAYS = 8
N_SPLITS     = 3
DATA_DIR     = './'   # final_input 파일 위치

FIN_V1_COLS = [
    'return_1d','return_3d','return_5d','volatility_3d',
    'volume_change','volume_ma_ratio','sector_return_mean',
    'relative_return_to_sector','relative_return_to_market',
    'relative_volatility_to_sector_3d'
]
FIN_V2_COLS = [
    'return_1d','return_3d','return_5d','volatility_5d',
    'volume_change','volume_ma_ratio','sector_return_mean',
    'relative_return_to_sector','relative_return_to_market',
    'relative_volatility_to_sector_5d'
]
MKT_COLS = [
    'VIX_Close','VIX_return_1d','VIX_change_3d',
    'SPY_return_1d','SPY_return_3d','QQQ_return_3d',
    'Oil_return_1d','Gold_return_1d','Dollar_return_1d','Treasury10Y_return_1d'
]
LABEL_COLS = {
    '0.5%': 'label_05',
    '0.7%': 'label_07',
    '1.0%': 'label_10',
}

In [ ]:
# ── 라벨 계산 (Close 가격 → 익일 수익률 방향) ──────────
raw = pd.read_csv(DATA_DIR + 'israel_hamas_FIN_MKT_features(step3).csv')
raw.columns = raw.columns.str.strip()
raw['Date'] = pd.to_datetime(raw['Date'])
raw = raw.sort_values(['ticker','Date']).reset_index(drop=True)
raw['next_return'] = raw.groupby('ticker')['Close'].pct_change(1).shift(-1)

for thr, name in [(0.005,'label_05'), (0.007,'label_07'), (0.01,'label_10')]:
    raw[name] = np.nan
    raw.loc[raw['next_return'] >  thr, name] = 2
    raw.loc[raw['next_return'] < -thr, name] = 0
    raw.loc[(raw['next_return'] >= -thr) & (raw['next_return'] <= thr), name] = 1

labels = raw[['Date','ticker','label_05','label_07','label_10']]

for col in ['label_05','label_07','label_10']:
    counts = labels[col].value_counts().sort_index()
    total  = counts.sum()
    print(f'{col}  down:{counts.get(0,0)}({counts.get(0,0)/total*100:.0f}%)  '
          f'neutral:{counts.get(1,0)}({counts.get(1,0)/total*100:.0f}%)  '
          f'up:{counts.get(2,0)}({counts.get(2,0)/total*100:.0f}%)  '
          f'NaN:{labels[col].isna().sum()}')

In [ ]:
def run_experiment(df, feat_cols, label_col):
    """TimeSeriesSplit CV(날짜 단위) + Holdout 평가.
    (cv_f1, holdout_f1, n_train, n_holdout) 반환
    - 일반 k-fold와 달리 학습 구간이 항상 검증 구간보다 과거임을 보장.
    - 같은 날짜의 종목이 학습/검증에 섞이지 않도록 '날짜' 단위로 split.
    """
    sub = df[feat_cols + [label_col, 'Date']].dropna(subset=[label_col])
    sub = sub.dropna(subset=feat_cols, how='all')

    dates         = sorted(sub['Date'].unique())
    holdout_dates = dates[-HOLDOUT_DAYS:]
    train_dates   = dates[:-HOLDOUT_DAYS]

    train = sub[sub['Date'].isin(train_dates)].copy()
    hold  = sub[sub['Date'].isin(holdout_dates)].copy()

    X_ho = hold[feat_cols].values
    y_ho = hold[label_col].values.astype(int)

    model = HistGradientBoostingClassifier(
        learning_rate=0.05, max_depth=4,
        min_samples_leaf=20, class_weight='balanced',
        random_state=42
    )

    # ── TimeSeriesSplit: '날짜' 배열에 대해 분할 → 같은 날 종목 묶음 유지 ──
    train_dates_sorted = sorted(train['Date'].unique())
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)

    cv_scores = []
    for tr_date_idx, val_date_idx in tscv.split(train_dates_sorted):
        tr_dates  = [train_dates_sorted[i] for i in tr_date_idx]
        val_dates = [train_dates_sorted[i] for i in val_date_idx]

        tr_mask  = train['Date'].isin(tr_dates)
        val_mask = train['Date'].isin(val_dates)

        X_tr_cv = train.loc[tr_mask,  feat_cols].values
        y_tr_cv = train.loc[tr_mask,  label_col].values.astype(int)
        X_val   = train.loc[val_mask, feat_cols].values
        y_val   = train.loc[val_mask, label_col].values.astype(int)

        # 검증 구간에 클래스가 1개뿐이면 스킵 (작은 데이터 대비)
        if len(np.unique(y_tr_cv)) < 2:
            continue

        model.fit(X_tr_cv, y_tr_cv)
        pred = model.predict(X_val)
        cv_scores.append(f1_score(y_val, pred, average='macro', zero_division=0))

    cv_mean = np.mean(cv_scores) if cv_scores else np.nan

    # ── Holdout: train 전체로 학습 후 1회 평가 ──
    X_tr = train[feat_cols].values
    y_tr = train[label_col].values.astype(int)
    model.fit(X_tr, y_tr)
    ho_pred = model.predict(X_ho)
    ho_f1   = f1_score(y_ho, ho_pred, average='macro', zero_division=0)

    return round(cv_mean, 4), round(ho_f1, 4), len(train), len(hold)

In [ ]:
# ── 전체 Ablation 실행 ────────────────────────────────
results = []
groups_map = ['FIN', 'MKT', 'EMO', 'ECON']

for vol_ver, fin_cols, fname in [
    ('vol3d', FIN_V1_COLS, 'model_input_dataset_3day_volatility(step3).csv'),
    ('vol5d', FIN_V2_COLS, 'model_input_dataset_5day_volatility(step3).csv'),
]:
    print(f'\n{"="*55}')
    print(f'FIN 버전: {vol_ver}')
    print(f'{"="*55}')

    df = pd.read_csv(DATA_DIR + fname, parse_dates=['Date'])

    # 라벨 merge
    df = pd.merge(df, labels, on=['Date','ticker'], how='left')

    emo_cols  = [c for c in df.columns if c.startswith('emo_')]
    econ_cols = [c for c in df.columns if c.startswith('reddit_')]
    id_cols  = [c for c in df.columns if (c.startswith('sector_') or c.startswith('ticker_')) and c != 'sector_return_mean']

    col_map = {
        'FIN' : fin_cols,
        'MKT' : MKT_COLS,
        'EMO' : emo_cols,
        'ECON': econ_cols,
    }

    for r in range(1, 5):
        for combo in itertools.combinations(groups_map, r):
            feat_cols  = id_cols.copy()
            for g in combo:
                feat_cols += col_map[g]
            combo_name = '+'.join(combo)

            for thr_name, label_col in LABEL_COLS.items():
                cv_f1, ho_f1, n_tr, n_ho = run_experiment(df, feat_cols, label_col)
                results.append({
                    'fin_ver'   : vol_ver,
                    'threshold' : thr_name,
                    'groups'    : combo_name,
                    'n_features': len(feat_cols),
                    'cv_f1'     : cv_f1,
                    'holdout_f1': ho_f1,
                    'n_train'   : n_tr,
                    'n_holdout' : n_ho,
                })
                print(f'[{vol_ver}] {thr_name} | ID+{combo_name:<22} | CV:{cv_f1:.3f}  Holdout:{ho_f1:.3f}')

In [ ]:
# ── 결과 저장 ─────────────────────────────────────────
result_df = pd.DataFrame(results)
result_df = result_df.sort_values(['fin_ver','threshold','holdout_f1'], ascending=[True,True,False])
result_df.to_csv(DATA_DIR + 'ablation_results_tscv.csv', index=False)
print('→ ablation_results_tscv.csv 저장 완료')

In [ ]:
# ── 요약: 임계값별 Holdout Top 5 ─────────────────────
for thr in ['0.5%', '0.7%', '1.0%']:
    sub = result_df[result_df['threshold'] == thr]
    print(f'\n★ [{thr}] Holdout F1 Top 5')
    print(sub.nlargest(5, 'holdout_f1')[['fin_ver','groups','cv_f1','holdout_f1']].to_string(index=False))

In [ ]:
# ── 피벗 테이블: 한눈에 보기 (임계값 선택) ───────────
TARGET_THR = '0.7%'   # ← 0.5% / 0.7% / 1.0% 중 선택

pivot = result_df[result_df['threshold'] == TARGET_THR].pivot_table(
    index='groups', columns='fin_ver',
    values='holdout_f1'
).round(3)

pivot = pivot.sort_values('vol3d', ascending=False)
print(f'[{TARGET_THR}] Holdout F1 — 피처 조합 × FIN버전')
print(pivot.to_string())